# 02 — Designing a causal transaction graph

We rejected all-pairs cliques because `(card1, addr1)` would produce ~145 million undirected pair links. Instead, we construct **at most 20 historical predecessors within 30 days** per relation.

A transaction at time `t` receives a directed edge from another only if source time is **strictly less** than `t`. Exact time ties remain unordered. Row index is the local PyG node ID, obtained by sorting `TransactionDT`, then `TransactionID`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
print("Repository:", ROOT)

In [ ]:
import numpy as np
import pandas as pd
import torch
from fraud_graph.config import GraphConfig
from fraud_graph.graph import chronological_frame, build_temporal_edges, union_edges, validate_temporal_edges
from fraud_graph.paths import RAW_DIR, PROCESSED_DIR

transaction = pd.read_csv(RAW_DIR / "train_transaction.csv", usecols=[
    "TransactionID", "TransactionDT", "card1", "addr1"
])
identity = pd.read_csv(RAW_DIR / "train_identity.csv", usecols=["TransactionID", "DeviceInfo"])
df = chronological_frame(transaction.merge(identity, on="TransactionID", how="left", validate="one_to_one"))
print(df.shape)
if (PROCESSED_DIR / "node_transaction_ids.npy").exists():
    assert np.array_equal(df.TransactionID.to_numpy(), np.load(PROCESSED_DIR / "node_transaction_ids.npy"))

## Graph behavior with K and W

- `W=30` controls staleness. In original EDA, about 96.5% of eligible consecutive same-key transactions occurred within 30 days.
- `K=20` caps neighbor explosion. V1 address graph yielded 6,393,246 edges.
- Combined deduplicated union can have **up to 40** incoming edges because each relation contributes up to 20.

In [ ]:
cfg = GraphConfig(window_days=30, max_neighbors=20)
edge_addr = build_temporal_edges(df, ("card1", "addr1"), cfg)
print("Address", validate_temporal_edges(edge_addr, df, cfg, max_in_degree=20))
edge_device = build_temporal_edges(df, ("card1", "DeviceInfo"), cfg)
print("Device", validate_temporal_edges(edge_device, df, cfg, max_in_degree=20))
edge_union = union_edges(edge_addr, edge_device, num_nodes=len(df))
print("Combined", validate_temporal_edges(edge_union, df, cfg, max_in_degree=40))

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
torch.save(edge_addr, PROCESSED_DIR / "edge_index_v1.pt")
torch.save(edge_device, PROCESSED_DIR / "edge_index_device_v1.pt")
torch.save(edge_union, PROCESSED_DIR / "edge_index_combined_v1.pt")
print("Saved three topology files")